# Swarm-Optimized Neural Network Ensembles
**Multi-Agent Systems — Final Project**

This notebook is a thin driver. All logic lives in the `.py` modules on Drive.
The only cell you should need to edit between runs is **Section 3 — Configuration**.

---
**Sections**
1. Environment Setup
2. GPU Check
3. Configuration — edit this between runs
4. Smoke Test
5. Full Ablation Run
6. Results & Plots
7. Hyperparameter Sweep (optional)
8. Topology Sweep — varies n and k together
9. k Sweep — holds n=12, varies k to isolate neighborhood size effect

## 1 — Environment Setup

In [1]:
import subprocess, os, sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    REPO_URL = 'https://github.com/blakedehaas/multi-agent-ml.git'
    REPO_DIR = '/content/multi-agent-ml'
    BRANCH   = 'implementation'

    if not os.path.exists(REPO_DIR):
        subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH], check=True)

    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
else:
    # Local: notebook is in notebooks/, repo root is one level up
    REPO_DIR = str(Path('..').resolve())
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
    subprocess.run(['pip', 'install', '-q', 'ipywidgets'], check=True)
    import warnings
    import numpy as np
    warnings.filterwarnings('ignore', message='.*align.*')
    warnings.filterwarnings('ignore', category=np.exceptions.VisibleDeprecationWarning)




print(f"Environment : {'Colab' if IN_COLAB else 'local'}")
print(f"Repo        : {REPO_DIR}")

Environment : local
Repo        : /Users/oscarrodriguez/Documents/Multi-Agent_Systems/Final_Project



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
# Pull latest changes from GitHub — run any time there is an update mid-session.
# On Colab: pulls from remote. Locally: use git pull in your terminal instead.
if IN_COLAB:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH], check=True)
    print('Pulled latest changes.')
else:
    print('Local run — use git pull in terminal to update.')

Local run — use git pull in terminal to update.


In [3]:
from config import DRIVE_ROOT, CHECKPOINT_DIR, DATA_ROOT, DEVICE, IN_COLAB

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Drive root   : {DRIVE_ROOT}')
print(f'Checkpoints  : {CHECKPOINT_DIR}')
print(f'Data root    : {DATA_ROOT}')
print(f'Device       : {DEVICE}')

Drive root   : /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/Final_Project
Checkpoints  : /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/Final_Project/experiments/checkpoints
Data root    : /Users/oscarrodriguez/Documents/Multi-Agent_Systems/Final_Project/data
Device       : mps


In [4]:
if IN_COLAB:
    subprocess.run(['pip', 'install', '-q', '--upgrade', 'ipython'], check=True)
    subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
else:
    print('Local run — installation skipped. Ensure your venv is active.')

Local run — installation skipped. Ensure your venv is active.


In [5]:
%load_ext autoreload
%autoreload 2
print('Autoreload enabled.')

Autoreload enabled.


In [6]:
# Log in to W&B — paste your API key when prompted
# Get your key at: https://wandb.ai/settings
import wandb
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/oscarrodriguez/.netrc.
wandb: Currently logged in as: osro6012 (osro6012-university-of-colorado-boulder) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## 2 — GPU Check

In [7]:
import torch

print(f'Device       : {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU          : {torch.cuda.get_device_name(0)}')
    print(f'VRAM         : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print(f'PyTorch      : {torch.__version__}  CUDA: {torch.version.cuda}')
elif DEVICE == 'mps':
    print('Apple MPS -- M-series GPU via Metal')
    print(f'PyTorch      : {torch.__version__}')
else:
    print('WARNING: running on CPU -- training will be slow.')
    print(f'PyTorch      : {torch.__version__}')

Device       : mps
Apple MPS -- M-series GPU via Metal
PyTorch      : 2.11.0


## 3 — Configuration

**This is the only cell you need to edit between runs.**

Key knobs:
- `conditions` — which ablation cells to run (`None` = all 8)
- `epochs` — 50 is a good default; increase to 100 for a final run
- `alpha`, `beta`, `gamma` — rule strengths when active
- `k` — neighborhood size (3 = sparse, 9 = full connectivity for N=10)

In [8]:
from experiments.ablation import AblationConfig

cfg = AblationConfig(
    # ── Experiment identity — shows as a group in W&B ─────────────────
    experiment_tag = 'ablation_v1',   # change this for each new experiment

    # ── Dataset ───────────────────────────────────────────────────────
    dataset = 'cifar10',     # 'mnist' for quick tests, 'cifar10' for real runs

    # ── Swarm rule strengths ──────────────────────────────────────────
    alpha = 0.3,     # gradient alignment
    beta  = 0.5,     # separation  -> equilibrium spacing = beta/gamma = 5.0
    gamma = 0.1,     # cohesion
    k     = 4,       # k-NN neighborhood size

    # ── Training ─────────────────────────────────────────────────────
    n_agents    = 12,
    epochs      = 50,
    batch_size  = 512,
    subset_size = None,     # None = full dataset; use 5000 for local smoke tests
    device      = DEVICE,
    lr          = 1e-3,
    weight_decay= 1e-4,

    # ── Logging ───────────────────────────────────────────────────────
    cka_interval     = 5,
    landscape_at_end = True,
    wandb_project    = 'swarm-optimization',
    wandb_mode       = 'online',

    # ── Conditions ───────────────────────────────────────────────────
    # None = run all 8. Or specify a subset:
    # conditions = ['baseline', 'separation', 'full_swarm']
    conditions = None,

    # ── Paths ─────────────────────────────────────────────────────────
    checkpoint_dir = CHECKPOINT_DIR,
)

print(f'Experiment   : {cfg.experiment_tag}')
print(f'Dataset      : {cfg.dataset}')
print(f'Conditions   : {cfg.conditions or "all 8"}')
print(f'Agents       : {cfg.n_agents}')
print(f'Epochs       : {cfg.epochs}')
print(f'Batch size   : {cfg.batch_size}')
print(f'Dataset size : {"full" if cfg.subset_size is None else cfg.subset_size}')
print(f'Rules        : alpha={cfg.alpha}  beta={cfg.beta}  gamma={cfg.gamma}  k={cfg.k}')
print(f'W&B mode     : {cfg.wandb_mode}')
print(f'Checkpoints  : {cfg.checkpoint_dir}')

Experiment   : ablation_v1
Dataset      : cifar10
Conditions   : all 8
Agents       : 12
Epochs       : 50
Batch size   : 512
Dataset size : full
Rules        : alpha=0.3  beta=0.5  gamma=0.1  k=4
W&B mode     : online
Checkpoints  : /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/Final_Project/experiments/checkpoints


## 3.5 — Throughput Profile

Measures forward+backward throughput across batch sizes on the current device.
Run this once per machine to find the batch size where GPU utilization plateaus.

**How to read it:** `samp/s` should increase as batch size grows and then level off.
The point where it levels off is the sweet spot — beyond it you pay memory cost
with no throughput gain and gradients become noisier.
`Est epoch` is the projected training time per epoch with 12 agents on full CIFAR-10.

In [9]:
import time
import torch
import torch.nn as nn
from models.cnn import TinyNet
from baselines.single_trainer import AgentTrainer, TrainingConfig

BATCH_SIZES = [64, 128, 256, 512, 1024, 2048]
N_WARMUP    = 5
N_TIMED     = 80

# Projected full-run parameters for the epoch estimate column
_N_AGENTS_FULL = 12
_N_TRAIN_FULL  = 45000   # 90% of CIFAR-10 50k train set after val split

def _device_sync():
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
    elif DEVICE == 'mps':
        torch.mps.synchronize()

torch.manual_seed(0)
_train_cfg = TrainingConfig(device=DEVICE, lr=1e-3)
_agent     = AgentTrainer(TinyNet(), _train_cfg)
_criterion = nn.CrossEntropyLoss()

print(f'device : {DEVICE}')
print(f'model  : TinyNet ({_agent.model.param_count():,} params)')
print(f'warmup : {N_WARMUP} batches   timed : {N_TIMED} batches\n')

print(f'{"batch":>8}  {"samp/s":>12}  {"ms/batch":>10}  {"ms/samp":>9}  {"est epoch (12 agents)":>22}')
print('─' * 70)

for bs in BATCH_SIZES:
    X = torch.randn(bs, 3, 32, 32, device=DEVICE)
    y = torch.randint(0, 10, (bs,), device=DEVICE)

    # Warmup — lets MPS/CUDA JIT compile the graph before timing
    for _ in range(N_WARMUP):
        _agent.optimizer.zero_grad()
        _criterion(_agent.model(X), y).backward()
        _agent.optimizer.step()
    _device_sync()

    t0 = time.perf_counter()
    for _ in range(N_TIMED):
        _agent.optimizer.zero_grad()
        _criterion(_agent.model(X), y).backward()
        _agent.optimizer.step()
    _device_sync()
    elapsed = time.perf_counter() - t0

    ms_per_batch  = elapsed / N_TIMED * 1000
    throughput    = bs * N_TIMED / elapsed
    ms_per_sample = ms_per_batch / bs
    epoch_sec     = (ms_per_batch / 1000) * (_N_TRAIN_FULL / bs) * _N_AGENTS_FULL
    epoch_str     = f'{epoch_sec / 60:.1f} min' if epoch_sec >= 60 else f'{epoch_sec:.0f} s'

    print(f'{bs:>8}  {throughput:>12,.0f}  {ms_per_batch:>10.1f}  {ms_per_sample:>9.3f}  {epoch_str:>22}')

print('\nUpdate cfg.batch_size above if a different value gives clearly better throughput.')

device : mps
model  : TinyNet (94,762 params)
warmup : 5 batches   timed : 80 batches

   batch        samp/s    ms/batch    ms/samp   est epoch (12 agents)
──────────────────────────────────────────────────────────────────────
      64        13,022         4.9      0.077                    41 s
     128        16,047         8.0      0.062                    34 s
     256        16,597        15.4      0.060                    33 s
     512        13,548        37.8      0.074                    40 s
    1024        13,656        75.0      0.073                    40 s
    2048        13,592       150.7      0.074                    40 s

Update cfg.batch_size above if a different value gives clearly better throughput.


## 4 — Smoke Test

Run 3 agents x 2 conditions x 3 epochs before committing to the full run.
Exercises the complete pipeline: training, CKA, loss landscape, PCA plots, and comparison table.

**Expected output:** two conditions complete, summary table prints, no errors.

In [14]:
from experiments.ablation import run_ablation, AblationConfig

smoke_cfg = AblationConfig(
    dataset        = cfg.dataset,
    n_agents       = 10,
    epochs         = 5,
    subset_size    = 40000,
    device         = DEVICE,
    cka_interval   = 5,
    landscape_at_end      = True,   # must be True — exercises plot_agent_pca path
    landscape_grid_size   = 5,      # tiny grid, fast
    landscape_alpha_range = (-0.5, 0.5),
    wandb_mode     = 'disabled',
    checkpoint_dir = cfg.checkpoint_dir / 'smoke',
    conditions     = ['baseline', 'full_swarm'],
    # inherit rule strengths and k from main cfg
    alpha = cfg.alpha,
    beta  = cfg.beta,
    gamma = cfg.gamma,
    k     = cfg.k,
)

smoke_results = run_ablation(smoke_cfg)
print('\nSmoke test passed — safe to proceed with full run.')

Running 2 ablation condition(s): ['baseline', 'full_swarm']

────────────────────────────────────────────────────────────
Condition: baseline  (α=0, β=0, γ=0)
────────────────────────────────────────────────────────────

Run: baseline
Trainer: <baselines.single_trainer.IndependentEnsembleTrainer object at 0x130729e00>
Agents: 10 × TinyNet (94,762 params each)
Train batches: 71  Val batches: 8



Training:   0%|          | 0/5 [00:00<?, ?epoch/s]

  Epoch   1:   0%|          | 0/71 [00:00<?, ?batch/s]

Epoch   0  train=2.0151  val=1.8051  ens=1.7438  acc=0.369


  Epoch   2:   0%|          | 0/71 [00:00<?, ?batch/s]

...

Epoch   2  train=1.4887  val=1.4281  ens=1.3904  acc=0.514


  Epoch   4:   0%|          | 0/71 [00:00<?, ?batch/s]

Epoch   3  train=1.3922  val=1.3623  ens=1.3080  acc=0.550


  Epoch   5:   0%|          | 0/71 [00:00<?, ?batch/s]

Epoch   4  train=1.3094  val=1.2954  ens=1.2538  acc=0.555

Test  ensemble=1.2553  acc=0.553  best=1.2660  diversity=16.0713
Checkpoint saved → /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/Final_Project/experiments/checkpoints/smoke/full_swarm.pt

────────────────────────────────────────────────────────────
Computing loss landscapes (shared color scale)...
  Grid: baseline
  Grid: full_swarm
  Landscape saved → /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/Final_Project/experiments/checkpoints/smoke/baseline_landscape.png
  PCA saved → /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/Final_Project/experiments/checkpoints/smoke/baseline_pca.png
  Landscape saved → /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/Final_Project/experiments/checkpoints/smoke/full_swarm_landscape.png
  PCA saved → /Users/oscarrodriguez/Library/CloudStorag

## 5 — Full Ablation Run

Trains all 8 conditions (2^3 combinations of alignment, separation, cohesion) sequentially.
On Colab A100: metrics stream live to W&B. Locally with MPS: wandb_mode is offline.

Estimated time on MPS (50 epochs, 12 agents, full CIFAR-10): ~1 hour per condition, ~8 hours total.
Estimated time on A100 (same config): ~30 min per condition, ~4 hours total.
Landscape computation (landscape_at_end=True) adds ~10-15 min after training completes.

In [11]:
# Patch default data roots so datasets download to local Colab storage
# rather than inside the repo clone. This avoids re-downloading on git pull.
import data.cifar as cifar_module
import data.mnist as mnist_module

cifar_module._DEFAULT_DATA_ROOT = DATA_ROOT / 'cifar10'
mnist_module._DEFAULT_DATA_ROOT = DATA_ROOT / 'mnist'

results = run_ablation(cfg)

Running 8 ablation condition(s): ['baseline', 'alignment', 'separation', 'cohesion', 'align_sep', 'align_coh', 'sep_coh', 'full_swarm']

────────────────────────────────────────────────────────────
Condition: baseline  (α=0, β=0, γ=0)
────────────────────────────────────────────────────────────



Run: baseline
Trainer: <baselines.single_trainer.IndependentEnsembleTrainer object at 0x12ea52350>
Agents: 12 × TinyNet (94,762 params each)
Train batches: 88  Val batches: 10



Training:   0%|          | 0/50 [00:00<?, ?epoch/s]

  Epoch   1:   0%|          | 0/88 [00:00<?, ?batch/s]

Epoch   0  train=1.9589  val=1.7403  ens=1.6802  acc=0.390


...

  Landscape saved → /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/Final_Project/experiments/checkpoints/align_sep_landscape.png
  PCA saved → /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/Final_Project/experiments/checkpoints/align_sep_pca.png


  Landscape saved → /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/Final_Project/experiments/checkpoints/align_coh_landscape.png
  PCA saved → /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/Final_Project/experiments/checkpoints/align_coh_pca.png


  Landscape saved → /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/Final_Project/experiments/checkpoints/sep_coh_landscape.png
  PCA saved → /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/Final_Project/experiments/checkpoints/sep_coh_pca.png


  Landscape saved → /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/Final_Project/experiments/checkpoints/full_swarm_landscape.png
  PCA saved → /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/Final_Project/experiments/checkpoints/full_swarm_pca.png



Ablation complete.

Condition        Ens Loss    Ens Acc   Ens F1   Best Acc  Diversity    GAP CKA
──────────────────────────────────────────────────────────────────────────────
baseline           0.6744      0.761    0.758      0.743      22.43      0.907
alignment          0.6790      0.761    0.758      0.745      20.59      0.904
separation         0.7551      0.737    0.737      0.708     132.02      0.877
cohesion           0.7526      0.736    0.735      0.738       9.63      0.946
align_sep          0.7601      0.730    0.730      0.699     129.50      0.864
align_coh          0.7082      0.749    0.746      0.734       8.53      0.952
sep_coh            0.6966      0.758    0.755      0.744      17.14      0.914
full_swarm         0.7058      0.753    0.749      0.742      17.09      0.907


## 6 — Results & Plots

In [12]:
# Summary comparison table
from experiments.ablation import compare_conditions
compare_conditions(results)


Condition        Ens Loss    Ens Acc   Ens F1   Best Acc  Diversity    GAP CKA
──────────────────────────────────────────────────────────────────────────────
baseline           0.6744      0.761    0.758      0.743      22.43      0.907
alignment          0.6790      0.761    0.758      0.745      20.59      0.904
separation         0.7551      0.737    0.737      0.708     132.02      0.877
cohesion           0.7526      0.736    0.735      0.738       9.63      0.946
align_sep          0.7601      0.730    0.730      0.699     129.50      0.864
align_coh          0.7082      0.749    0.746      0.734       8.53      0.952
sep_coh            0.6966      0.758    0.755      0.744      17.14      0.914
full_swarm         0.7058      0.753    0.749      0.742      17.09      0.907


In [13]:
import matplotlib.pyplot as plt
from visualization.plots import plot_diversity_curves, plot_training_curves

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for ax, (label, res) in zip(axes, results.items()):
    if res['cka_history']:
        fig_cka = plot_diversity_curves(res['cka_history'])
        src_ax = fig_cka.axes[0]
        for line in src_ax.lines:
            ax.plot(line.get_xdata(), line.get_ydata(),
                    label=line.get_label(), linewidth=line.get_linewidth())
        ax.set_title(label, fontsize=10)
        ax.set_ylim(0, 1.05)
        ax.set_xlabel('Epoch', fontsize=8)
        ax.set_ylabel('Mean CKA', fontsize=8)
        ax.grid(True, alpha=0.3)
        plt.close(fig_cka)

fig.suptitle('Representational Diversity by Condition', fontsize=13)
fig.tight_layout()
cfg.checkpoint_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(str(cfg.checkpoint_dir / 'diversity_curves.png'), dpi=150)
plt.show()

/var/folders/_0/b8gbhj617mzdpkh9flk3q1kc0000gn/T/ipykernel_98658/236395713.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
# ── Training loss curves side by side ─────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for ax, (label, res) in zip(axes, results.items()):
    epochs_  = [d['epoch']     for d in res['history']]
    means    = [d['mean_loss'] for d in res['history']]
    ax.plot(epochs_, means, linewidth=2, color='black', label='mean')
    for i in range(len(res['history'][0]['agent_losses'])):
        agent_curve = [d['agent_losses'][i] for d in res['history']]
        ax.plot(epochs_, agent_curve, linewidth=0.8, alpha=0.4)
    ax.set_title(label, fontsize=10)
    ax.set_xlabel('Epoch', fontsize=8)
    ax.set_ylabel('Loss', fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle('Training Loss by Condition', fontsize=13)
fig.tight_layout()
plt.savefig(str(cfg.checkpoint_dir / 'training_curves.png'), dpi=150)
plt.show()

/var/folders/_0/b8gbhj617mzdpkh9flk3q1kc0000gn/T/ipykernel_98658/2019244195.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [15]:
# ── Final CKA matrix for each condition at gap layer ──────────────────────
import matplotlib.colors as mcolors
from visualization.plots import plot_cka_matrix

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.flatten()

for ax, (label, res) in zip(axes, results.items()):
    if res['cka_history']:
        last   = res['cka_history'][-1]
        matrix = last['gap']['matrix']
        fig_cka = plot_cka_matrix(matrix, layer='gap', epoch=last['epoch'])
        src_ax  = fig_cka.axes[0]
        img     = src_ax.images[0]
        ax.imshow(img.get_array(), vmin=0, vmax=1, cmap='viridis', aspect='auto')
        ax.set_title(label, fontsize=10)
        ax.axis('off')
        plt.close(fig_cka)

sm = plt.cm.ScalarMappable(
    cmap='viridis',
    norm=mcolors.Normalize(vmin=0, vmax=1),
)
sm.set_array([])
fig.colorbar(sm, ax=axes.tolist(), label='CKA Similarity', shrink=0.6, pad=0.02)

fig.suptitle('Final CKA Similarity (GAP layer)', fontsize=13)
fig.tight_layout()
plt.savefig(str(cfg.checkpoint_dir / 'cka_matrices.png'), dpi=150, bbox_inches='tight')
plt.show()

/var/folders/_0/b8gbhj617mzdpkh9flk3q1kc0000gn/T/ipykernel_98658/362149741.py:28: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()
/var/folders/_0/b8gbhj617mzdpkh9flk3q1kc0000gn/T/ipykernel_98658/362149741.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7 — Hyperparameter Sweep (Optional)

Run this section only after the ablation has identified a winning condition.
The sweep fine-tunes the rule strengths for that condition using Bayesian optimization.

**Step 1** creates the sweep on W&B (run once).
**Step 2** launches the agent (can duplicate this cell for parallel agents).

In [ ]:
# ── Step 1: Create sweep — run ONCE, then comment out ────────────────────
from experiments.hyperparam_sweep import create_sweep

SWEEP_CONDITION = 'full_swarm'   # ← change to your winning condition

sweep_id = create_sweep(
    condition = SWEEP_CONDITION,
    project   = 'swarm-optimization',
    method    = 'bayes',
    n_trials  = 30,
)
print(f'sweep_id = {sweep_id!r}  ← paste into Step 2')

In [ ]:
# ── Step 2: Launch agent ──────────────────────────────────────────────────
from experiments.hyperparam_sweep import run_sweep_agent

SWEEP_ID = ''   # ← paste sweep_id from Step 1

run_sweep_agent(
    sweep_id       = SWEEP_ID,
    condition      = SWEEP_CONDITION,
    project        = 'swarm-optimization',
    n_agents       = cfg.n_agents,
    epochs         = cfg.epochs,
    subset_size    = cfg.subset_size,
    device         = DEVICE,
    checkpoint_dir = CHECKPOINT_DIR / 'sweep',
    count          = 10,   # runs per agent process
)

## 8 — Topology Sweep

Tests whether the ratio of neighborhood size (k) to population size (n_agents) affects training stability.
Runs only `sep_coh` and `full_swarm` — the two conditions that showed instability in the full ablation.

| Config | n | k | k divides n? | k/n ratio |
|--------|---|---|--------------|-----------|
| n9_k3  | 9 | 3 | yes | 0.33 |
| n10_k3 | 10 | 3 | no  | 0.30 (original) |
| n12_k3 | 12 | 3 | yes | 0.25 |
| n12_k4 | 12 | 4 | yes | 0.33 |

Hypothesis: if instability disappears when k divides n, graph asymmetry is the cause.
If instability persists across all configs, the beta/gamma force imbalance is the cause.

In [16]:
# ── Topology sweep: does k dividing n_agents affect stability? ───────────
# Runs only sep_coh + full_swarm (the unstable conditions) across 4 (n, k) configs.
# alpha, beta, gamma are unchanged from the main ablation.

from experiments.ablation import run_ablation, AblationConfig

topology_configs = [
    dict(n_agents=9,  k=3, label='n9_k3'),   # k divides n: yes  ratio=0.33
    dict(n_agents=10, k=3, label='n10_k3'),  # k divides n: no   ratio=0.30  (original)
    dict(n_agents=12, k=3, label='n12_k3'),  # k divides n: yes  ratio=0.25
    dict(n_agents=12, k=4, label='n12_k4'),  # k divides n: yes  ratio=0.33
]

topology_results = {}

for tc in topology_configs:
    print(f'\n{"="*60}')
    print(f'Running: n_agents={tc["n_agents"]}, k={tc["k"]}  ({tc["label"]})')
    print(f'{"="*60}')

    topo_cfg = AblationConfig(
        experiment_tag = f'topology_{tc["label"]}',

        dataset = cfg.dataset,
        alpha = cfg.alpha,
        beta  = cfg.beta,
        gamma = cfg.gamma,
        k     = tc['k'],

        n_agents    = tc['n_agents'],
        epochs      = cfg.epochs,
        batch_size  = cfg.batch_size,
        subset_size = cfg.subset_size,
        device      = DEVICE,
        lr          = cfg.lr,
        weight_decay= cfg.weight_decay,

        conditions       = ['sep_coh', 'full_swarm'],
        landscape_at_end = False,
        cka_interval     = 5,
        wandb_mode       = cfg.wandb_mode,
        wandb_project    = cfg.wandb_project,
        checkpoint_dir   = cfg.checkpoint_dir / 'topology_sweep' / tc['label'],
    )

    res = run_ablation(topo_cfg)
    topology_results[tc['label']] = {
        'n': tc['n_agents'],
        'k': tc['k'],
        'results': res,
    }

print('\nTopology sweep complete.')


Running: n_agents=9, k=3  (n9_k3)
Running 2 ablation condition(s): ['sep_coh', 'full_swarm']

────────────────────────────────────────────────────────────
Condition: sep_coh  (α=0, β=0.5, γ=0.1)
────────────────────────────────────────────────────────────



Run: sep_coh
Trainer: SwarmTrainer(n_agents=9, α=0, β=0.5, γ=0.1, k=3)
Agents: 9 × TinyNet (94,762 params each)
Train batches: 88  Val batches: 10



Training:   0%|          | 0/50 [00:00<?, ?epoch/s]

  Epoch   1:   0%|          | 0/88 [00:00<?, ?batch/s]

Epoch   0  train=1.9581  val=1.7449  ens=1.6885  acc=0.381


...

Epoch  48  train=0.7109  val=0.7784  ens=0.7036  acc=0.762


  Epoch  50:   0%|          | 0/88 [00:00<?, ?batch/s]

Epoch  49  train=0.7090  val=0.7738  ens=0.6977  acc=0.756

Test  ensemble=0.7058  acc=0.753  best=0.7470  diversity=17.0937
Checkpoint saved → /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/Final_Project/experiments/checkpoints/topology_sweep/n12_k4/full_swarm.pt


cka/block1/max_sim,▆█▇▄▃▂▂▁▁▁
cka/block1/mean_sim,▁▅▆▆▇███▇▆
cka/block1/min_sim,▁▆▇▇▇█▇█▇▇
cka/block2/max_sim,█▆▆▂▂▂▂▁▁▁
cka/block2/mean_sim,█▆▆▄▃▃▂▂▂▁
cka/block2/min_sim,▂██▇▅▄▃▁▁▁
cka/block3/max_sim,█▅▄▄▃▂▂▂▂▁
cka/block3/mean_sim,█▅▄▄▄▃▂▂▂▁
cka/block3/min_sim,▄▇█▇▇▄▅▃▂▁
cka/gap/max_sim,█▆▅▄▄▃▂▂▂▁
+33,...



Ablation complete.

Condition        Ens Loss    Ens Acc   Ens F1   Best Acc  Diversity    GAP CKA
──────────────────────────────────────────────────────────────────────────────
sep_coh            0.6966      0.758    0.755      0.744      17.14      0.914
full_swarm         0.7058      0.753    0.749      0.742      17.09      0.907

Topology sweep complete.


In [17]:
# ── Topology sweep results table ─────────────────────────────────────────

print(f'\n{"Config":<10} {"Condition":<12} {"n":>4} {"k":>4} {"k/n":>6} '
      f'{"k div n?":>9} {"Ens Acc":>8} {"Ens F1":>8} {"Diversity":>10} {"GAP CKA":>8}')
print('─' * 83)

for label, entry in topology_results.items():
    n, k = entry['n'], entry['k']
    divides = 'yes' if n % k == 0 else 'no'
    ratio   = k / n

    for cond, res in entry['results'].items():
        tm      = res['test_metrics']
        gap_cka = '—'
        if res['cka_history']:
            last = res['cka_history'][-1]
            if 'gap' in last:
                gap_cka = f'{last["gap"]["mean_sim"]:.3f}'

        print(f'{label:<10} {cond:<12} {n:>4} {k:>4} {ratio:>6.2f} '
              f'{divides:>9} {tm["ensemble_acc"]:>8.3f} {tm["ensemble_f1"]:>8.3f} '
              f'{tm["diversity"]:>10.2f} {gap_cka:>8}')
    print()


Config     Condition       n    k    k/n  k div n?  Ens Acc   Ens F1  Diversity  GAP CKA
───────────────────────────────────────────────────────────────────────────────────
n9_k3      sep_coh         9    3   0.33       yes    0.749    0.749      15.04    0.836
n9_k3      full_swarm      9    3   0.33       yes    0.701    0.698      14.65    0.911

n10_k3     sep_coh        10    3   0.30        no    0.745    0.741      19.21    0.792
n10_k3     full_swarm     10    3   0.30        no    0.745    0.745      16.11    0.865

n12_k3     sep_coh        12    3   0.25       yes    0.749    0.747      19.33    0.802
n12_k3     full_swarm     12    3   0.25       yes    0.732    0.728      16.04    0.883

n12_k4     sep_coh        12    4   0.33       yes    0.758    0.755      17.14    0.914
n12_k4     full_swarm     12    4   0.33       yes    0.753    0.749      17.09    0.907



In [18]:
# ── Training curves: spot instability visually ────────────────────────────

import matplotlib.pyplot as plt

conditions  = ['sep_coh', 'full_swarm']
n_configs   = len(topology_results)

fig, axes = plt.subplots(len(conditions), n_configs,
                          figsize=(5 * n_configs, 4 * len(conditions)),
                          sharey='row')

for col, (label, entry) in enumerate(topology_results.items()):
    n, k = entry['n'], entry['k']
    divides = 'yes' if n % k == 0 else 'no'

    for row, cond in enumerate(conditions):
        ax  = axes[row, col]
        res = entry['results'].get(cond)

        if res and res['history']:
            epochs_  = [d['epoch']     for d in res['history']]
            means    = [d['mean_loss'] for d in res['history']]

            for i in range(len(res['history'][0]['agent_losses'])):
                agent_curve = [d['agent_losses'][i] for d in res['history']]
                ax.plot(epochs_, agent_curve, linewidth=0.8, alpha=0.4)

            ax.plot(epochs_, means, linewidth=2, color='black', label='mean')

        if row == 0:
            ax.set_title(f'{label}\nn={n}, k={k}, k div n={divides}', fontsize=10)
        if col == 0:
            ax.set_ylabel(f'{cond}\nLoss', fontsize=9)
        ax.set_xlabel('Epoch', fontsize=8)
        ax.grid(True, alpha=0.3)
        ax.set_ylim(0.5, 2.1)

out_dir = cfg.checkpoint_dir / 'topology_sweep'
out_dir.mkdir(parents=True, exist_ok=True)
fig.suptitle('Topology Sweep — Training Stability (sep_coh & full_swarm)', fontsize=13)
fig.tight_layout()
plt.savefig(str(out_dir / 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

/var/folders/_0/b8gbhj617mzdpkh9flk3q1kc0000gn/T/ipykernel_98658/183480011.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9 — k Sweep (n=12 fixed)

Isolates the effect of neighborhood size by holding n=12 constant and varying k.
The topology sweep showed k=3 is unstable and k=4 is stable — this sweep fills in the gap
and extends to near full-connectivity to map the full stability profile.

| k  | k divides 12? | k/n  | Notes                        |
|----|---------------|------|------------------------------|
| 3  | yes           | 0.25 | Known unstable               |
| 4  | yes           | 0.33 | Known stable                 |
| 5  | no            | 0.42 | Non-divisor                  |
| 6  | yes           | 0.50 | Half the swarm               |
| 8  | no            | 0.67 | Non-divisor, dense           |
| 9  | yes           | 0.75 | Three-quarters               |
| 11 | no            | 0.92 | Full connectivity (k = n-1)  |

If stability tracks divisibility: k=5, 8, 11 should be unstable; k=6, 9 stable.
If stability tracks absolute k: expect a threshold somewhere between k=3 and k=4, monotone above it.
k=11 is the all-to-all reference — symmetric graph, in-degree equals k for all agents.

In [19]:
# ── k sweep: n=12 fixed, k varies ────────────────────────────────────────
# Runs sep_coh + full_swarm for each k value.
# alpha, beta, gamma unchanged from main ablation.

from experiments.ablation import run_ablation, AblationConfig

N_AGENTS = 12
k_values = [3, 4, 5, 6, 8, 9, 11]

k_sweep_results = {}

for k in k_values:
    label = f'n12_k{k}'
    print(f'\n{"="*60}')
    print(f'Running: n_agents={N_AGENTS}, k={k}  ({label})')
    print(f'{"="*60}')

    k_cfg = AblationConfig(
        experiment_tag = f'k_sweep_{label}',

        dataset = cfg.dataset,
        alpha = cfg.alpha,
        beta  = cfg.beta,
        gamma = cfg.gamma,
        k     = k,

        n_agents    = N_AGENTS,
        epochs      = cfg.epochs,
        batch_size  = cfg.batch_size,
        subset_size = cfg.subset_size,
        device      = DEVICE,
        lr          = cfg.lr,
        weight_decay= cfg.weight_decay,

        conditions       = ['sep_coh', 'full_swarm'],
        landscape_at_end = False,
        cka_interval     = 5,
        wandb_mode       = cfg.wandb_mode,
        wandb_project    = cfg.wandb_project,
        checkpoint_dir   = cfg.checkpoint_dir / 'k_sweep' / label,
    )

    res = run_ablation(k_cfg)
    k_sweep_results[label] = {
        'n': N_AGENTS,
        'k': k,
        'results': res,
    }

print('\nk sweep complete.')


Running: n_agents=12, k=3  (n12_k3)
Running 2 ablation condition(s): ['sep_coh', 'full_swarm']

────────────────────────────────────────────────────────────
Condition: sep_coh  (α=0, β=0.5, γ=0.1)
────────────────────────────────────────────────────────────



Run: sep_coh
Trainer: SwarmTrainer(n_agents=12, α=0, β=0.5, γ=0.1, k=3)
Agents: 12 × TinyNet (94,762 params each)
Train batches: 88  Val batches: 10



Training:   0%|          | 0/50 [00:00<?, ?epoch/s]

  Epoch   1:   0%|          | 0/88 [00:00<?, ?batch/s]

Epoch   0  train=1.9583  val=1.7391  ens=1.6803  acc=0.389


...

Epoch  48  train=0.7330  val=0.8223  ens=0.7435  acc=0.745


  Epoch  50:   0%|          | 0/88 [00:00<?, ?batch/s]

Epoch  49  train=0.7294  val=0.8153  ens=0.7377  acc=0.747

Test  ensemble=0.7507  acc=0.737  best=0.8011  diversity=40.6401
Checkpoint saved → /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/Final_Project/experiments/checkpoints/k_sweep/n12_k11/full_swarm.pt


cka/block1/max_sim,▇▅▇█▇▇▅▄▁▁
cka/block1/mean_sim,▁▇▅▄▄▃▅▄█▇
cka/block1/min_sim,▁▅▅▆▇▆▇▇██
cka/block2/max_sim,█▅▄▄▃▂▂▂▁▁
cka/block2/mean_sim,█▆▄▃▂▂▁▁▁▁
cka/block2/min_sim,█▇▅▂▂▁▁▂▂▁
cka/block3/max_sim,█▄▃▃▃▃▂▂▁▁
cka/block3/mean_sim,█▅▄▃▃▂▂▂▁▁
cka/block3/min_sim,█▇▇▅▅▃▂▁▂▁
cka/gap/max_sim,█▅▄▄▃▃▂▂▂▁
+33,...



Ablation complete.

Condition        Ens Loss    Ens Acc   Ens F1   Best Acc  Diversity    GAP CKA
──────────────────────────────────────────────────────────────────────────────
sep_coh            0.7495      0.738    0.735      0.725      40.74      0.896
full_swarm         0.7507      0.737    0.735      0.720      40.64      0.890

k sweep complete.


In [20]:
# ── k sweep results table ─────────────────────────────────────────────────

print(f'\n{"Config":<10} {"Condition":<12} {"k":>4} {"k/n":>6} '
      f'{"k div 12?":>10} {"Ens Acc":>8} {"Ens F1":>8} {"Diversity":>10} {"GAP CKA":>8}')
print('─' * 80)

for label, entry in k_sweep_results.items():
    n, k = entry['n'], entry['k']
    divides = 'yes' if n % k == 0 else 'no'
    ratio   = k / n

    for cond, res in entry['results'].items():
        tm      = res['test_metrics']
        gap_cka = '—'
        if res['cka_history']:
            last = res['cka_history'][-1]
            if 'gap' in last:
                gap_cka = f'{last["gap"]["mean_sim"]:.3f}'

        print(f'{label:<10} {cond:<12} {k:>4} {ratio:>6.2f} '
              f'{divides:>10} {tm["ensemble_acc"]:>8.3f} {tm["ensemble_f1"]:>8.3f} '
              f'{tm["diversity"]:>10.2f} {gap_cka:>8}')
    print()


Config     Condition       k    k/n  k div 12?  Ens Acc   Ens F1  Diversity  GAP CKA
────────────────────────────────────────────────────────────────────────────────
n12_k3     sep_coh         3   0.25        yes    0.749    0.747      19.33    0.802
n12_k3     full_swarm      3   0.25        yes    0.732    0.728      16.04    0.883

n12_k4     sep_coh         4   0.33        yes    0.758    0.755      17.14    0.914
n12_k4     full_swarm      4   0.33        yes    0.753    0.749      17.09    0.907

n12_k5     sep_coh         5   0.42         no    0.751    0.747      19.92    0.913
n12_k5     full_swarm      5   0.42         no    0.751    0.746      20.29    0.909

n12_k6     sep_coh         6   0.50        yes    0.750    0.746      23.29    0.909
n12_k6     full_swarm      6   0.50        yes    0.751    0.746      23.34    0.906

n12_k8     sep_coh         8   0.67         no    0.743    0.738      30.24    0.902
n12_k8     full_swarm      8   0.67         no    0.742    0.739

In [21]:
# ── k sweep training curves ───────────────────────────────────────────────

import matplotlib.pyplot as plt

conditions = ['sep_coh', 'full_swarm']
n_configs  = len(k_sweep_results)

fig, axes = plt.subplots(len(conditions), n_configs,
                          figsize=(4 * n_configs, 4 * len(conditions)),
                          sharey='row')

for col, (label, entry) in enumerate(k_sweep_results.items()):
    k       = entry['k']
    divides = 'yes' if entry['n'] % k == 0 else 'no'

    for row, cond in enumerate(conditions):
        ax  = axes[row, col]
        res = entry['results'].get(cond)

        if res and res['history']:
            epochs_  = [d['epoch']     for d in res['history']]
            means    = [d['mean_loss'] for d in res['history']]

            for i in range(len(res['history'][0]['agent_losses'])):
                agent_curve = [d['agent_losses'][i] for d in res['history']]
                ax.plot(epochs_, agent_curve, linewidth=0.8, alpha=0.4)

            ax.plot(epochs_, means, linewidth=2, color='black', label='mean')

        if row == 0:
            ax.set_title(f'k={k}  (div={divides})', fontsize=10)
        if col == 0:
            ax.set_ylabel(f'{cond}\nLoss', fontsize=9)
        ax.set_xlabel('Epoch', fontsize=8)
        ax.grid(True, alpha=0.3)
        ax.set_ylim(0.5, 2.1)

out_dir = cfg.checkpoint_dir / 'k_sweep'
out_dir.mkdir(parents=True, exist_ok=True)
fig.suptitle('k Sweep (n=12) — Training Stability (sep_coh & full_swarm)', fontsize=13)
fig.tight_layout()
plt.savefig(str(out_dir / 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

/var/folders/_0/b8gbhj617mzdpkh9flk3q1kc0000gn/T/ipykernel_98658/2573029278.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
